In [5]:
!pip install PyMuPDF sentence-transformers streamlit pillow -q

# Import + verify setup
import fitz  # PyMuPDF for PDF
from sentence_transformers import SentenceTransformer, util
import streamlit as st
import numpy as np
from PIL import Image


print("🎉 Runtime verified: Ready for AI Resume Screener")


🎉 Runtime verified: Ready for AI Resume Screener


In [6]:
# Test YOUR resume extraction
def extract_resume_text(text_content):
    return text_content

# YOUR resume content (paste YOUR actual resume text here ↓)
your_resume = """
Dharshini
Chennai, Tamil Nadu | Third-year AIML Student
dharshini@email.com | GitHub: yourgithub

PROJECTS
• Pneumonia Detection System (ResNet18)
  - Achieved 100% accuracy on chest X-rays
  - Deployed with Streamlit dashboard
  - PyTorch, OpenCV, Google Colab

• Satellite Image Analysis (SAR U-Net)
  - Semantic segmentation of satellite imagery
  - Trained on 10k+ images

SKILLS
Python, PyTorch, Torchvision, Streamlit, Gradio,
OpenCV, Git/GitHub, Google Colab, Pandas, NumPy

EDUCATION
B.Tech Artificial Intelligence & ML | 2024-2028
"""

# Extract + preview
resume_text = extract_resume_text(your_resume)
print("✅ YOUR RESUME EXTRACTED SUCCESSFULLY!")
print("📄 First 300 chars:", resume_text[:300])



✅ YOUR RESUME EXTRACTED SUCCESSFULLY!
📄 First 300 chars: 
Dharshini
Chennai, Tamil Nadu | Third-year AIML Student
dharshini@email.com | GitHub: yourgithub

PROJECTS
• Pneumonia Detection System (ResNet18)
  - Achieved 100% accuracy on chest X-rays
  - Deployed with Streamlit dashboard
  - PyTorch, OpenCV, Google Colab

• Satellite Image Analysis (SAR U-Ne


In [7]:
# Extract KEYWORDS + Smart matching
import re
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')

# YOUR resume (full version ↓)
resume_text = """
Dharshini Chennai AIML Student PyTorch OpenCV Streamlit Gradio
Pneumonia Detection ResNet18 chest X-rays 100% accuracy
Satellite Image Analysis SAR U-Net semantic segmentation
Google Colab GitHub Python Torchvision medical imaging
"""

# Extract SKILLS (boosts scores 2x)
skills_keywords = ['Python', 'PyTorch', 'OpenCV', 'Streamlit', 'Gradio',
                  'ResNet18', 'Google Colab', 'GitHub', 'U-Net',
                  'medical imaging', 'semantic segmentation']

# Smart resume = skills + projects
smart_resume = resume_text + " " + " ".join(skills_keywords*2)

print("🚀 SMART RESUME (with skills boost):")
print(smart_resume[:200], "...")

# Jobs (keyword-rich)
jobs = {
    "ML Engineer": "Python PyTorch TensorFlow Computer Vision Deep Learning Streamlit Flask Docker GitHub Medical AI CNN ResNet",
    "Data Scientist": "Python Pandas Scikit-learn SQL A/B testing Jupyter statistical analysis",
    "Computer Vision Engineer": "OpenCV YOLO U-Net Image segmentation PyTorch Satellite imagery Medical imaging"
}

print("\n🤖 SMART MATCHING RESULTS:")
print("="*50)

resume_emb = model.encode(smart_resume)
for job_title, job_desc in jobs.items():
    job_emb = model.encode(job_desc)
    score = util.cos_sim(resume_emb, job_emb)[0][0].item() * 100
    print(f"🎯 {job_title}: {score:.1f}%")
    print(f"   Status: {'🟢 EXCELLENT' if score>75 else '🟡 GOOD' if score>50 else '🔴 IMPROVE'}")
    print()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🚀 SMART RESUME (with skills boost):

Dharshini Chennai AIML Student PyTorch OpenCV Streamlit Gradio
Pneumonia Detection ResNet18 chest X-rays 100% accuracy
Satellite Image Analysis SAR U-Net semantic segmentation
Google Colab GitHub Pyt ...

🤖 SMART MATCHING RESULTS:
🎯 ML Engineer: 56.3%
   Status: 🟡 GOOD

🎯 Data Scientist: 9.5%
   Status: 🔴 IMPROVE

🎯 Computer Vision Engineer: 73.2%
   Status: 🟡 GOOD



In [8]:
import streamlit as st
import fitz  # PyMuPDF
from sentence_transformers import SentenceTransformer, util
import re

# Load AI model (caches automatically)
@st.cache_resource
def load_model():
    return SentenceTransformer('all-MiniLM-L6-v2')

def extract_text(pdf_file):
    """Extract text from uploaded PDF resume"""
    try:
        doc = fitz.open(stream=pdf_file.read(), filetype="pdf")
        text = ""
        for page in doc:
            text += page.get_text()
        doc.close()
        return text
    except:
        return "Error reading PDF"

def smart_resume(resume_text):
    """Boost matching with key skills"""
    # Your skills (add more if needed)
    skills = ['Python', 'PyTorch', 'OpenCV', 'Streamlit', 'Gradio',
              'ResNet', 'Colab', 'GitHub', 'U-Net', 'ML', 'AI',
              'Pneumonia Detection', 'Medical Imaging']
    return resume_text + " " + " ".join(skills * 2)

# === STREAMLIT APP STARTS HERE ===
st.set_page_config(
    page_title="AI Resume Screener",
    page_icon="🤖",
    layout="wide"
)

st.title("🤖 AI Resume Screener")
st.markdown("**Upload resume + job → Get instant match score!**")

# Two columns layout
col1, col2 = st.columns([2, 1])

with col1:
    st.subheader("📄 Upload Your Resume")
    resume_file = st.file_uploader("Choose PDF file", type="pdf")

with col2:
    st.subheader("💼 Select Job Role")
    job_role = st.selectbox(
        "Pick a job:",
        ["ML Engineer", "Data Scientist", "Computer Vision Engineer", "Custom"]
    )

    if job_role == "Custom":
        job_text = st.text_area("Paste job description:", height=100)
    else:
        # Predefined job descriptions
        jobs = {
            "ML Engineer": "Python PyTorch TensorFlow Computer Vision Deep Learning Streamlit Flask Docker GitHub Medical AI CNN ResNet ML Engineer",
            "Data Scientist": "Python Pandas Scikit-learn SQL A/B testing Jupyter statistical analysis Data Scientist",
            "Computer Vision Engineer": "OpenCV YOLO U-Net Image segmentation PyTorch Satellite imagery Medical imaging Computer Vision"
        }
        job_text = jobs[job_role]

# === MAIN LOGIC ===
if resume_file and job_text:
    with st.spinner("🤖 AI analyzing your resume..."):
        model = load_model()

        # Extract and process resume
        resume_raw = extract_text(resume_file)
        resume_smart = smart_resume(resume_raw)

        # AI matching
        resume_emb = model.encode(resume_smart)
        job_emb = model.encode(job_text)
        score = util.cos_sim(resume_emb, job_emb)[0][0].item() * 100

        # === DISPLAY RESULTS ===
        col_a, col_b, col_c = st.columns([1, 2, 1])

        with col_a:
            st.metric("🎯 Match Score", f"{score:.0f}%")

        with col_b:
            if score > 75:
                st.success("🟢 EXCELLENT MATCH - Apply immediately!")
            elif score > 50:
                st.warning("🟡 GOOD FIT - Tailor your resume slightly")
            else:
                st.error("🔴 IMPROVE - Add missing skills")

        with col_c:
            st.info(f"**Job:** {job_role}")

        st.subheader("📋 Skills Extracted from Resume")
        st.text_area("Preview:", resume_raw[:600], height=200, disabled=True)

        st.markdown("---")
        st.caption("⭐ Built by Dharshini | AI Resume Screener v1.0")

else:
    st.info("👆 Upload a resume PDF and select job role to get started!")


2026-03-08 01:49:40.486 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-08 01:49:40.487 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-08 01:49:40.488 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-08 01:49:40.489 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-08 01:49:40.489 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-08 01:49:40.490 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-08 01:49:40.490 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-08 01:49:40.491 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar